# Circular Linked List

The last node points back to the first, so there is no `None` terminator and
traversal wraps around. Useful for round-robin scheduling and ring buffers,
where "next" should never run out.

Because the list has no end to fall off, every node can reach every other node.
Each insert below appears twice: once as the obvious O(n) version that walks to
the last node, and once in O(1) using a value-swap trick that avoids the walk
entirely.

We'll use dummy head for all functions.

![Circular Linked List](images/circular-linked-list.png)

### Node and traversal helper

The node is identical to a singly linked list's -- one `next` pointer. Only the
*shape* differs: the last node points back at the first instead of at `None`.

That breaks the usual `while curr:` loop, which would spin forever. Every traversal
must instead remember where it began and stop when it wraps back around, which is
what `to_list` does with `while curr is not first`.

In [ ]:
class ListNode:
    def __init__(self, val=0):
        self.val = val
        self.next = None

def to_list(head):
    ls = []
    first = head.next
    if not first: # empty list
        return ls
    ls.append(first.val)
    curr = first.next
    while curr is not first: # continue till list wraps around
        ls.append(curr.val)
        curr = curr.next
    return ls

### Insert at beginning, the direct way

Reaching the first node is easy (`head.next`), but making a *new* node the first node
means the **last** node's `next` has to be repointed at it -- and finding the last
node means walking the entire ring.

```
D → [1] → [2] → [3] ┐        insert 9 at the beginning
     ↑______________│

walk until curr.next is first   →  curr = [3]
new.next = curr.next            →  9 → [1]
curr.next = new                 →  [3] → 9
head.next = new                 →  9 is now first

D → [9] → [1] → [2] → [3] ┐
```

**Time:** O(n), essentially all of it spent finding the last node &nbsp;
**Space:** O(1)

In [ ]:
def insert_begin_linear(head, val):
    new = ListNode(val)
    first = head.next # keep reference to first node for wraparound check
    if not first: # empty list, new node becomes the head
        new.next = new # make circular
    else:
        curr = first.next
        while curr.next is not first: # find the last node
            curr = curr.next
        # insert new node
        new.next = curr.next # link new node to the last node
        curr.next = new
    head.next = new # update head to new node


def test_insert_begin_linear():
    head = ListNode(-1)
    insert_begin_linear(head, 1)
    assert to_list(head) == [1]
    insert_begin_linear(head, 2)
    assert to_list(head) == [2, 1]
    insert_begin_linear(head, 3)
    assert to_list(head) == [3, 2, 1]

test_insert_begin_linear()

### Insert at beginning in O(1)

The trick: don't move nodes, move **values**. Insert the new node in *second*
position -- which needs no walk, since the first node is right there -- then swap the
two values so the new value ends up in the node that is already first.

```
insert 9 into  D → [1] → [2] → …

link after the first node:   D → [1] → [9] → [2] → …
swap the two values:         D → [9] → [1] → [2] → …
```

`head` never changes, so the last node is never needed. The catch: the node holding a
given value changes, so any reference a caller kept to the old first node now sees a
different value. Fine for a value-only list, wrong if node identity matters.

**Time:** O(1) &nbsp; **Space:** O(1)

In [ ]:
def insert_begin_constant(head, val):
    # neat trick, insert new node at second place and swap data with first node
    new = ListNode(val)
    first = head.next # keep reference to first node for wraparound check
    if not first: #empty list
        new.next = new # make circular
        head.next = new # update head to new node
    else:
        # add new node after first node
        new.next = first.next
        first.next = new
        new.val, first.val = first.val, new.val # swap data
        # no need to update head as first node is still the first node


def test_insert_begin_constant():
    head = ListNode(-1)
    insert_begin_constant(head, 1)
    assert to_list(head) == [1]
    insert_begin_constant(head, 2)
    assert to_list(head) == [2, 1]
    insert_begin_constant(head, 3)
    assert to_list(head) == [3, 2, 1]

test_insert_begin_constant()

### Insert at end, the direct way

The same walk as the linear insert-at-beginning, with one difference: `head` is left
alone. The new node is spliced in after the last node, so it becomes the new last
node rather than the new first.

Insert-at-beginning and insert-at-end differ *only* in whether `head` moves -- in a
ring there is no other distinction between the two ends.

**Time:** O(n) &nbsp; **Space:** O(1)

In [ ]:
def insert_end_linear(head, val):
    new = ListNode(val)
    first = head.next # keep reference to first node for wraparound check
    if not first: # empty list
        new.next = new # make circular
        head.next = new # update head to new node
    else:
        curr = first.next
        while curr.next is not first: # find the last node
            curr = curr.next
        # insert new node after the last
        new.next = curr.next # link new node to the last node
        curr.next = new


def test_insert_end_linear():
    head = ListNode(-1)
    insert_end_linear(head, 1)
    assert to_list(head) == [1]
    insert_end_linear(head, 2)
    assert to_list(head) == [1, 2]
    insert_end_linear(head, 3)
    assert to_list(head) == [1, 2, 3]

test_insert_end_linear()

### Insert at end in O(1)

The same value-swap trick, plus one extra move. Insert second and swap values -- that
leaves the *old* first value sitting in the second node. Then point `head` at that
second node, which re-labels the ring: the node holding the new value is now the one
just before `head`'s target, i.e. the last.

```
insert 9 into  D → [1] → [2] → [3] ┐

link new node after the first:   [1] → [9] → [2] → [3] ┐
swap the two values:             [9] → [1] → [2] → [3] ┐
head.next = the second node:  D → [1] → [2] → [3] → [9] ┐

no walk, and 9 is last
```

Both constant-time variants work only because a ring has no inherent end -- `head`
alone decides which node counts as first.

**Time:** O(1) &nbsp; **Space:** O(1)

In [ ]:
def insert_end_constant(head, val):
    # neat trick, insert new node at second place, swap data with head and new node becomes head
    new = ListNode(val)
    first = head.next # keep reference to first node for wraparound check
    if not first: # empty list
        new.next = new # make circular
        head.next = new # update head to new node
    else:
        # add new node after first node
        new.next = first.next
        first.next = new
        new.val, first.val = first.val, new.val # swap data
        head.next = new # update head to new node


def test_insert_end_constant():
    head = ListNode(-1)
    insert_end_constant(head, 1)
    assert to_list(head) == [1]
    insert_end_constant(head, 2)
    assert to_list(head) == [1, 2]
    insert_end_constant(head, 3)
    assert to_list(head) == [1, 2, 3]

test_insert_end_constant()